the 2022 senate csv has the same columns as the house one, but PartyAB is blank. fill it in from PartyNm using data/aec_parties.csv (canonical names + aliases) and write the csv back so nb 02 doesnt have to deal with any of this. plain pandas, no spark.

load the inputs and take a peek.

In [1]:
import pandas as pd

SENATE_CSV  = '../data/2022_senate_candidates.csv'
PARTIES_CSV = '../data/aec_parties.csv'

aec_parties = pd.read_csv(PARTIES_CSV, header=0)
senate      = pd.read_csv(SENATE_CSV,  header=0)

print('AEC parties:', len(aec_parties))
print('Senate rows:', len(senate))
senate.head()

AEC parties: 38
Senate rows: 421


,state,PartyAB,PartyNm,Surname,GivenNm
0,ACT,NaN,Australian Labor Party,GALLAGHER,Katy
1,ACT,NaN,Australian Labor Party,NORTHAM,Maddy
2,ACT,NaN,Sustainable Australia Party - Stop Overdevelop...,ANGEL,Joy
3,ACT,NaN,Sustainable Australia Party - Stop Overdevelop...,HAYDON,John
4,ACT,NaN,United Australia Party,SAVOULIDIS,James


build a name → party_ab lookup. every party_name and every alias from the pipe-separated aliases column maps to the row's party_ab. lowercased so we can match case-insensitively.

In [2]:
name_to_party_ab = {}
for _, row in aec_parties.iterrows():
    ab = row['party_ab']
    names = [row['party_name']]
    if isinstance(row['aliases'], str) and row['aliases'].strip():
        names.extend(row['aliases'].split('|'))
    for n in names:
        if isinstance(n, str) and n.strip():
            name_to_party_ab[n.strip().lower()] = ab

print('Lookup entries:', len(name_to_party_ab))

Lookup entries: 55


now resolve PartyAB for each senate row. rules in order. blank or NaN PartyNm goes to IND. exact (case-insensitive) match in the lookup goes to that party's PartyAb. personal-name ticket (no "party" in the name) goes to IND. otherwise leave it blank and dump it in the unmapped list for triage.

In [3]:
def resolve_party_ab(party_nm):
    if not isinstance(party_nm, str) or not party_nm.strip():
        return 'IND'
    key = party_nm.strip().lower()
    if key in name_to_party_ab:
        return name_to_party_ab[key]
    if 'party' not in key:
        return 'IND'
    return None  # genuinely unmapped

senate['PartyAB'] = senate['PartyNm'].apply(resolve_party_ab)

unmapped = senate.loc[senate['PartyAB'].isna(), 'PartyNm'].dropna().unique()
print('Filled rows:', senate['PartyAB'].notna().sum())
print('Unmapped rows:', senate['PartyAB'].isna().sum())
if len(unmapped):
    print('\nUnmapped PartyNm values (triage needed — add to aec_parties.csv aliases):')
    for v in sorted(unmapped):
        print(' -', v)

Filled rows: 421
Unmapped rows: 0


quick sanity check on the distribution.

write the senate csv back with PartyAB filled in. column order's preserved so nb 02 can just read it as-is.

In [6]:
senate.to_csv(SENATE_CSV, index=False)
print('Wrote:', SENATE_CSV)

Wrote: ../data/2022_senate_candidates.csv
